# مترجم خودکار مانگا / مانهوا به فارسی

OCR → پاک‌سازی حباب → ترجمه با Gemini → رندر فارسی

**ترتیب:** سلول‌ها را از بالا به پایین با Shift+Enter اجرا کنید.

قبل از شروع: `Runtime → Change runtime type → GPU (T4)`

In [ ]:
!git clone https://github.com/amirwolf512k/Manga-AutoTranslate.git
!cp Manga-AutoTranslate/manga_translator.py .
!rm -rf Manga-AutoTranslate

Telegram:
@Amir_wolf512
-------------------
حمایت مالی ( •̀ ω •́ )✧
Ton:UQBScvayaxagwTfRBhlLNaqw-sZuadlnBjSvn8OJz7XZJJzT
-------------------
TRX:TMmLTaCjaW1L2xWZmpR2EBeNyCawCzEkwa
-------------------


## 3) دانلود فونت‌های فارسی (Vazirmatn — چند وزن)

چند وزن دانلود می‌شود تا برای سبک‌های مختلف بالن فونت جدا استفاده شود:
- **Regular** → دیالوگ معمولی
- **ExtraBold / Black** → فریاد، انفجار، SFX
- **Light** → فکر / نجوا
- **Bold** → بالن مشکی


In [ ]:
import os
os.makedirs('fonts', exist_ok=True)
BASE = 'https://github.com/rastikerdar/vazirmatn/raw/master/fonts/ttf'
weights = ['Regular', 'Medium', 'SemiBold', 'Bold', 'ExtraBold', 'Black', 'Light']
for w in weights:
    path = f'fonts/Vazirmatn-{w}.ttf'
    if not os.path.isfile(path):
        !wget -q -O {path} {BASE}/Vazirmatn-{w}.ttf
    print(path, 'OK' if os.path.isfile(path) else 'FAIL', os.path.getsize(path) if os.path.isfile(path) else 0)

FONT_PATH = 'fonts/Vazirmatn-Regular.ttf'
FONT_SHOUT = 'fonts/Vazirmatn-ExtraBold.ttf'
FONT_EXPLOSION = 'fonts/Vazirmatn-Black.ttf'
FONT_SFX = 'fonts/Vazirmatn-Black.ttf'
FONT_BLACK = 'fonts/Vazirmatn-Bold.ttf'
FONT_THOUGHT = 'fonts/Vazirmatn-Light.ttf'
FONT_WHISPER = 'fonts/Vazirmatn-Light.ttf'
print('فونت‌ها آماده شد.')


## 4) انتخاب ارائه‌دهنده AI و کلید API

| Provider | کلید رایگان / لینک |
|----------|---------------------|
| **gemini** | [aistudio.google.com/api-keys](https://aistudio.google.com/api-keys) |
| **openai** | [platform.openai.com/api-keys](https://platform.openai.com/api-keys) |
| **deepseek** | [platform.deepseek.com](https://platform.deepseek.com/api_keys) |
| **groq** | [console.groq.com/keys](https://console.groq.com/keys) |
| **xai** | [console.x.ai](https://console.x.ai/) |
| **openrouter** | [openrouter.ai/keys](https://openrouter.ai/keys) |
| **ollama** | لوکال — در Colab معمولاً در دسترس نیست |


In [ ]:
from getpass import getpass
import os

print("ارائه‌دهنده AI را انتخاب کنید:")
print(" 1) gemini")
print(" 2) openai")
print(" 3) deepseek")
print(" 4) groq")
print(" 5) xai")
print(" 6) openrouter")
print(" 7) together")
choice = input("انتخاب [پیش‌فرض 1]: ").strip() or "1"
_map = {
    "1": "gemini", "2": "openai", "3": "deepseek",
    "4": "groq", "5": "xai", "6": "openrouter", "7": "together",
}
PROVIDER = _map.get(choice, "gemini")
print(f"→ provider = {PROVIDER}")

MODEL_NAME = input("مدل (Enter = پیش‌فرض provider): ").strip() or None
if MODEL_NAME:
    print(f"→ model = {MODEL_NAME}")

print("\nکلیدهای API را یکی‌یکی وارد کن (خالی بذار تا تموم بشه):")
keys = []
while True:
    k = getpass(f"کلید {len(keys)+1} (Enter = پایان): ").strip()
    if not k:
        break
    keys.append(k)

if not keys:
    raise SystemExit("حداقل یک کلید لازم است.")

API_KEYS = ",".join(keys)
os.environ["API_KEY"] = API_KEYS
_env_map = {
    "gemini": "GEMINI_API_KEY",
    "openai": "OPENAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "groq": "GROQ_API_KEY",
    "xai": "XAI_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "together": "TOGETHER_API_KEY",
}
os.environ[_env_map.get(PROVIDER, "API_KEY")] = API_KEYS
print(f"✅ {len(keys)} کلید برای {PROVIDER} ثبت شد.")

زبان اصلی متن منبع رو انتخاب کنید (این مهمه؛ انتخاب اشتباه باعث می‌شه OCR متن رو درست استخراج نکنه):

In [ ]:
print("زبان اصلی متن منبع رو انتخاب کنید:")
print(" 1) en (انگلیسی - اکثر اسکنلیشن‌ها)")
print(" 2) ja en (ژاپنی خام)")
print(" 3) ko en (کره‌ای خام)")
print(" 4) دستی وارد کنید")

lang_choice = input("انتخاب [پیش‌فرض 1]: ").strip() or "1"

if lang_choice == "1":
    OCR_LANG = "en"
elif lang_choice == "2":
    OCR_LANG = "ja en"
elif lang_choice == "3":
    OCR_LANG = "ko en"
elif lang_choice == "4":
    OCR_LANG = input("زبان OCR را وارد کنید: ").strip()
else:
    OCR_LANG = "en"

print(f"زبان OCR: {OCR_LANG}")
print()

## 5) ورودی رو بدید و ترجمه رو اجرا کنید
این سلول خودش تشخیص می‌ده که چی بهش دادید:
- اگه یک **لینک** (http/https) وارد کنید، تصاویر همون صفحه خودکار دانلود می‌شن.
- اگه Enter بزنید، پنجره‌ی آپلود باز می‌شه؛ می‌تونید یک فایل **.zip**، یک فایل **.pdf**، یا چند تا **تصویر** (jpg/png/...) رو هم‌زمان انتخاب کنید — نوعش خودکار تشخیص داده می‌شه.

In [ ]:
import os
from google.colab import files

INPUT_DIR = 'input_pages'
os.makedirs(INPUT_DIR, exist_ok=True)

url = input('اگه لینک صفحه دارید وارد کنید (وگرنه Enter بزنید تا فایل آپلود کنید): ').strip()

if url.lower().startswith('http://') or url.lower().startswith('https://'):
    input_path = url
    print(f'از لینک استفاده می‌شه: {input_path}')
else:
    print('فایل(ها) رو انتخاب کنید (یک zip، یک pdf، یا چند تصویر):')
    uploaded = files.upload()
    names = list(uploaded.keys())

    if len(names) == 1 and names[0].lower().endswith('.zip'):
        input_path = names[0]
        with open(input_path, 'wb') as f:
            f.write(uploaded[names[0]])
        print(f'فایل zip شناسایی و ذخیره شد: {input_path}')

    elif len(names) == 1 and names[0].lower().endswith('.pdf'):
        input_path = names[0]
        with open(input_path, 'wb') as f:
            f.write(uploaded[names[0]])
        print(f'فایل pdf شناسایی و ذخیره شد: {input_path}')

    else:
        for name, data in uploaded.items():
            with open(os.path.join(INPUT_DIR, name), 'wb') as f:
                f.write(data)
        input_path = INPUT_DIR
        print(f'{len(names)} تصویر آپلود و در پوشه‌ی {INPUT_DIR} ذخیره شد.')

print('\nورودی نهایی برای پردازش:', input_path)

خروجی پیش‌فرض **PDF** است و نام فایل خودکار از روی لینک یا فایل ورودی ساخته می‌شود (مثلاً `eu39-green-skin-chapter-13-eng-li.pdf`).

برای مانگای ژاپنی خام `--ocr-lang ja en`، برای کره‌ای `ko en`، برای انگلیسی اسکنلیشن فقط `en`.
برای کمیک چپ‌به‌راست، `--reading-order ltr` بگذارید.


In [ ]:
import os
import re
import zipfile
from urllib.parse import urlparse, unquote

use_wildcard = ("*" in input_path) or ("," in input_path)
out_ext = ".pdf"

def _auto_output_path(input_path: str, output_spec: str) -> str:
    spec = (output_spec or "").strip()
    is_ext_only = (
        spec.startswith(".")
        and "/" not in spec and "\\" not in spec
        and re.fullmatch(r"\.(pdf|zip|html)", spec, re.I) is not None
    )
    if not is_ext_only:
        return output_spec
    ext = spec.lower()
    if input_path.lower().startswith(("http://", "https://")):
        path = unquote(urlparse(input_path).path).strip("/")
        parts = [p for p in path.split("/") if p]
        base = "chapter"
        if parts:
            slug = parts[-1]
            m = re.search(
                r"(.+?-chapter[-_]?(?:\d+|\*))(?:[-_].*)?$",
                slug,
                flags=re.I,
            )
            if m:
                base = m.group(1)
            elif "chapter" in [p.lower() for p in parts]:
                low = [p.lower() for p in parts]
                try:
                    idx = low.index("chapter")
                    name = parts[idx - 1] if idx > 0 else "chapter"
                    num = parts[idx + 1] if idx + 1 < len(parts) else ""
                    num = re.sub(r"[^\w\-]", "", num.split("?")[0])
                    base = f"{name}-{num}" if num else name
                except ValueError:
                    base = slug
            else:
                base = slug
        base = re.sub(r"\*+", "", base)
        base = re.sub(r"[^\w\-.]+", "-", base)
        base = re.sub(r"-{2,}", "-", base).strip("-._") or "chapter"
    else:
        raw = input_path.rstrip("/\\")
        base = os.path.splitext(os.path.basename(raw))[0] or "output"
        base = re.sub(r"[^\w\-.]+", "-", base).strip("-._") or "output"
    return base + ext

expected = _auto_output_path(input_path, out_ext)
print(f"خروجی پیش‌بینی‌شده: {expected}")
if use_wildcard:
    print("[*] حالت چندفصل (* یا ,) → بعد از ترجمه، همه PDFها در یک ZIP جمع می‌شوند.")
print(PROVIDER, FONT_PATH)

# Colab روی ! چندخطی با \ متغیرها را خراب می‌کند → اول f-string، بعد !
cmd = (
    f"python manga_translator.py "
    f"-i {input_path!r} "
    f"-o {out_ext!r} "
    f"--font {FONT_PATH!r} "
    f"--font-shout {FONT_SHOUT!r} "
    f"--font-explosion {FONT_EXPLOSION!r} "
    f"--font-sfx {FONT_SFX!r} "
    f"--font-black {FONT_BLACK!r} "
    f"--font-thought {FONT_THOUGHT!r} "
    f"--font-whisper {FONT_WHISPER!r} "
    f"--ocr-lang {OCR_LANG!r} "
    f"--provider {PROVIDER!r} "
    f"--api-key {API_KEYS!r} "
    f"--reading-order rtl"
)
print(cmd[:120], "...")
!{cmd}

pdfs = [
    f for f in os.listdir(".")
    if f.lower().endswith(".pdf") and os.path.isfile(f)
]
series_key = re.sub(r"\*+", "", re.sub(r"[^\w\-.]+", "-",
    unquote(urlparse(input_path).path).strip("/").split("/")[-1] if input_path.lower().startswith("http") else input_path
))
series_key = re.sub(r"-{2,}", "-", series_key).strip("-._")
related = [f for f in pdfs if series_key and series_key.split("-chapter")[0][:12].lower() in f.lower()]
if not related:
    related = sorted(pdfs, key=lambda f: os.path.getmtime(f), reverse=True)

if use_wildcard and len(related) >= 1:
    zip_base = series_key
    zip_base = re.sub(r"-chapter-?\d*$", "-chapters", zip_base, flags=re.I)
    if not zip_base or zip_base == "chapter":
        zip_base = "chapters"
    zip_name = zip_base + ".zip"
    if os.path.isfile(zip_name):
        stem, n = zip_base, 2
        while os.path.isfile(f"{stem}-{n}.zip"):
            n += 1
        zip_name = f"{stem}-{n}.zip"

    related_sorted = sorted(related, key=lambda f: [
        int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", f)
    ])
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in related_sorted:
            zf.write(f, arcname=os.path.basename(f))
            print(f"  + {f}")
    output_path = zip_name
    print(f"[✓] {len(related_sorted)} فصل داخل ZIP ذخیره شد: {output_path}")
elif related:
    related.sort(key=lambda f: os.path.getmtime(f), reverse=True)
    output_path = related[0]
    print(f"فایل خروجی آماده است: {output_path}")
elif os.path.isfile(expected):
    output_path = expected
    print(f"فایل خروجی آماده است: {output_path}")
else:
    output_path = None
    print("هشدار: فایل خروجی پیدا نشد.")

## 6) دانلود خروجی‌ها

In [ ]:
from google.colab import files
import os
import re
import zipfile
import shutil
from IPython.display import display, IFrame
from PIL import Image as PILImage

def _natural_key(name: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", name)]

def _find_outputs():
    found = {"pdf": [], "zip": [], "html": []}
    for name in os.listdir("."):
        if not os.path.isfile(name):
            continue
        low = name.lower()
        if low.endswith(".pdf"):
            found["pdf"].append(name)
        elif low.endswith(".zip"):
            found["zip"].append(name)
        elif low.endswith(".html"):
            found["html"].append(name)
    for k in found:
        found[k].sort(key=lambda f: os.path.getmtime(f), reverse=True)
    return found

outputs = _find_outputs()
all_files = outputs["pdf"] + outputs["zip"] + outputs["html"]
if not all_files:
    raise SystemExit("هیچ فایل خروجی پیدا نشد. اول سلول ترجمه را اجرا کنید.")

if "output_path" in globals() and output_path and os.path.isfile(str(output_path)):
    primary = str(output_path)
else:
    primary = all_files[0]

pdfs_sorted = sorted(outputs["pdf"], key=_natural_key)
zips_sorted = sorted(outputs["zip"], key=_natural_key)
htmls_sorted = sorted(outputs["html"], key=_natural_key)
bundle = pdfs_sorted + zips_sorted + htmls_sorted

print(f"خروجی اصلی: {primary}")
if len(bundle) > 1:
    print(f"مجموع فایل‌های خروجی: {len(bundle)}")
    for i, f in enumerate(bundle, 1):
        mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {i}) {f}  ({mb:.1f} MB)")
else:
    print(f"حجم: {os.path.getsize(primary) / (1024 * 1024):.1f} MB")

print()
print("انتخاب کنید:")
print("  1) دانلود همه خروجی‌ها")
print("  2) نمایش ترجمه")
if len(bundle) > 1:
    print("  3) دانلود فقط یک فایل")
choice = (input("عدد را وارد کنید [پیش‌فرض 1]: ").strip() or "1")

if str(choice) == "1":
    if len(bundle) == 1:
        print(f"در حال دانلود: {bundle[0]}")
        files.download(bundle[0])
    else:
        pack_name = "translated_chapters.zip"
        n = 1
        while os.path.isfile(pack_name):
            n += 1
            pack_name = f"translated_chapters_{n}.zip"
        print(f"بسته‌بندی {len(bundle)} فایل → {pack_name}")
        with zipfile.ZipFile(pack_name, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in bundle:
                zf.write(f, arcname=os.path.basename(f))
                print(f"  + {f}")
        files.download(pack_name)

elif str(choice) == "3" and len(bundle) > 1:
    num = input(f"شماره فایل (1..{len(bundle)}): ").strip()
    try:
        idx = int(num) - 1
        if 0 <= idx < len(bundle):
            files.download(bundle[idx])
        else:
            print("شماره نامعتبر.")
    except ValueError:
        print("عدد وارد کنید.")

else:
    show_path = primary
    if len(bundle) > 1:
        print("\nکدام را نمایش بدهم؟")
        for i, f in enumerate(bundle, 1):
            print(f"  {i}) {f}")
        num = input("شماره [1]: ").strip() or "1"
        try:
            idx = int(num) - 1
            if 0 <= idx < len(bundle):
                show_path = bundle[idx]
        except ValueError:
            pass

    if not os.path.exists(show_path):
        print(f"فایل یافت نشد: {show_path}")

    elif show_path.lower().endswith(".html"):
        display(IFrame(src=show_path, width="100%", height=900))

    elif show_path.lower().endswith(".pdf"):
        try:
            display(IFrame(src=show_path, width="100%", height=900))
        except Exception as e:
            print(f"نمایش ممکن نشد ({e}). دانلود...")
            files.download(show_path)

    elif show_path.lower().endswith(".zip"):
        extract_folder = "temp_extracted_out"
        if os.path.isdir(extract_folder):
            shutil.rmtree(extract_folder, ignore_errors=True)
        os.makedirs(extract_folder, exist_ok=True)
        with zipfile.ZipFile(show_path, "r") as zf:
            zf.extractall(extract_folder)
            members = zf.namelist()

        pdfs, images = [], []
        img_exts = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
        for root, _, names in os.walk(extract_folder):
            for name in names:
                full = os.path.join(root, name)
                low = name.lower()
                if low.endswith(".pdf"):
                    pdfs.append(full)
                elif low.endswith(img_exts):
                    images.append(full)
        pdfs.sort(key=lambda p: _natural_key(os.path.basename(p)))
        images.sort(key=lambda p: _natural_key(os.path.basename(p)))

        if pdfs:
            for i, p in enumerate(pdfs, 1):
                print(f"  {i}) {os.path.basename(p)}")
            num = input("شماره فصل [1]: ").strip() or "1"
            try:
                idx = int(num) - 1
                target = pdfs[idx] if 0 <= idx < len(pdfs) else pdfs[0]
            except ValueError:
                target = pdfs[0]
            try:
                display(IFrame(src=target, width="100%", height=900))
            except Exception:
                files.download(target)
        elif images:
            for i, img_path in enumerate(images[:30], 1):
                print(f"[{i}] {os.path.basename(img_path)}")
                img = PILImage.open(img_path)
                if img.size[0] > 900:
                    img = img.resize((900, int(img.size[1] * 900 / img.size[0])), PILImage.LANCZOS)
                display(img)
        else:
            print("داخل ZIP چیزی برای نمایش نبود:", members[:20])
    else:
        files.download(show_path)